# 2. Obtención y almacenamiento de los datos

## Obtener los datos según las variables seleccionadas en el apartado anterior.

Se descargarán datos de distintas fuentes:

- Datos satelitales de Google Earth Engine (GEE):
  - Sentinel-2 (colección `COPERNICUS/S2_SR_HARMONIZED`) para calcular NDVI y NDBI,
    además de bandas ópticas B2, B4, B8, B11 y B12.
  - Landsat 8 (colección `LANDSAT/LC08/C02/T1_L2`) para obtener la temperatura superficial (LST).
  - Todo se exporta como un GeoTIFF multibanda para el área de Sevilla capital.

- Datos complementarios de OpenStreetMap (Overpass API):
  - Red viaria principal (`highway`: motorway, trunk, primary, secondary).
  - Edificios (`building`), incluyendo centro geométrico y altura estimada.

- Datos de arbolado urbano:
  - Inventario municipal de arbolado de Sevilla del ayuntamiento de Sevilla (id, especie y coordenadas).

Todos los archivos se guardan en `outputs/raw/` para su posterior procesamiento.

In [26]:
import os
import importlib
import src.descarga_datos as _mod
importlib.reload(_mod)
from src.descarga_datos import ejecutar_descarga
from src.config import load_config
import src.boto3_helper as boto3_helper
importlib.reload(boto3_helper)
from src.boto3_helper import upload_data_to_s3, setup_bucket, get_s3_client

In [25]:
path = os.path.join("..", "datasets", "raw")
ejecutar_descarga(path)

configuración cargada correctamente. Proyecto GEE: testworkflows-830d7
🛰️ Conectando con Earth Engine y filtrando colecciones...
⬇️ Descargando GeoTIFF desde Earth Engine...
✅ GeoTIFF descargado correctamente en: ../datasets/raw/sevilla_dataset.tif

 Descargando datos complementarios...
🌳 Descargando arbolado urbano desde OpenStreetMap (Overpass API)...
  · Consultando celda 1/16...
  · Consultando celda 2/16...
  · Consultando celda 3/16...
  · Consultando celda 4/16...
  · Consultando celda 5/16...
  · Consultando celda 6/16...
  · Consultando celda 7/16...
  · Consultando celda 8/16...
  · Consultando celda 9/16...
  · Consultando celda 10/16...
  · Consultando celda 11/16...
  · Consultando celda 12/16...
  · Consultando celda 13/16...
  · Consultando celda 14/16...
  · Consultando celda 15/16...
  · Consultando celda 16/16...
✅ Descarga completada: ../datasets/raw/arbolado_sevilla.csv (18054 árboles registrados)
🚗 Conectando a OpenStreetMap (Overpass API)...


HTTPError: 429 Client Error: Too Many Requests for url: http://overpass-api.de/api/interpreter?data=%0A++++%5Bout%3Ajson%5D%3B%0A++++%28%0A++++++way%5B%22highway%22~%22motorway%7Ctrunk%7Cprimary%7Csecondary%22%5D%2837.33%2C-6.03%2C37.45%2C-5.90%29%3B%0A++++%29%3B%0A++++out+center%3B+%0A++++

In [28]:
app_config = load_config()
    
cliente_s3 = get_s3_client(app_config)

setup_bucket(
    s3_client=cliente_s3, 
    bucket_name=app_config.s3_bucket_name, 
    region=app_config.aws_region
)

❌ Error: El nombre 'tfm-thermo-sevilla-raw-data-v1' ya está en uso por otro usuario de AWS.


In [4]:
archivos_ejemplo = [
    os.path.join("..", "datasets", "raw", "sevilla_dataset.tif"),
    os.path.join("..", "datasets", "raw", "arbolado_sevilla.csv"),
    os.path.join("..", "datasets", "raw", "carreteras_sevilla.csv"),
    os.path.join("..", "datasets", "raw", "edificios_sevilla.csv"),
]

for archivo in archivos_ejemplo:
    upload_data_to_s3(
        s3_client=cliente_s3, 
        file_path=archivo, 
        bucket_name=app_config.s3_bucket_name
    )

⏳ Subiendo '../datasets/raw/sevilla_dataset.tif' a 's3://tfm-thermo-sevilla-raw-data-v1/sevilla_dataset.tif'...
✅ Subida completada con éxito.
⏳ Subiendo '../datasets/raw/arbolado_sevilla.csv' a 's3://tfm-thermo-sevilla-raw-data-v1/arbolado_sevilla.csv'...
✅ Subida completada con éxito.
⏳ Subiendo '../datasets/raw/carreteras_sevilla.csv' a 's3://tfm-thermo-sevilla-raw-data-v1/carreteras_sevilla.csv'...
✅ Subida completada con éxito.
⏳ Subiendo '../datasets/raw/edificios_sevilla.csv' a 's3://tfm-thermo-sevilla-raw-data-v1/edificios_sevilla.csv'...
✅ Subida completada con éxito.


In [10]:
from src.boto3_helper import download_data_from_s3

In [11]:
download_data_from_s3(
    s3_client=cliente_s3, 
    bucket_name=app_config.s3_bucket_name, 
    object_name="arbolado_sevilla.csv", 
    local_file_path="../temp/datasets/raw/arbolado_sevilla_descargado.csv")

⏳ Descargando 's3://tfm-thermo-sevilla-raw-data-v1/arbolado_sevilla.csv' en '../temp/datasets/raw/arbolado_sevilla_descargado.csv'...
✅ Descarga completada con éxito.
